# Trading signals and execution timing

In [ ]:
from pathlib import Path
import sys

# Works whether Jupyter starts in the project root or in notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "config.json").exists() and (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data import read_config, download_data
config = read_config(ROOT)


In [ ]:
from src.indicators import add_features, trading_signals

data = download_data(ROOT, config)
ticker = "MU"
features = add_features(data[ticker], config)
signals = trading_signals(features)
table = features[["adj_close", "sma_fast", "sma_slow", "momentum"]].copy()
table["sma_signal_after_close"] = signals["SMA_20_50"]
table["target_at_next_open"] = signals["SMA_20_50"].shift(1)
table.loc["2025-06-02":"2025-06-13"].round(4)

In [ ]:
# Find actual changes in the desired open-time position.
position = signals["SMA_20_50"].shift(1)
changes = position.diff().fillna(0).ne(0)
table.loc[changes & (table.index >= "2025-06-02")].head(10).round(4)